# Unbiased AI Decision — Model Training + Fairness Metrics
### Phase 2: Train model, compute 5 bias metrics, generate fix recommendations

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings, os, json
warnings.filterwarnings('ignore')
matplotlib.use('Agg')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import joblib

from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equalized_odds_difference,
    selection_rate
)
from sklearn.metrics import recall_score, precision_score

os.makedirs('../models', exist_ok=True)
os.makedirs('../reports/charts', exist_ok=True)
os.makedirs('../reports', exist_ok=True)
print('All imports successful')

All imports successful


In [2]:
df = pd.read_csv('../data/ibm_hr_with_agegroup.csv')
print(f'Dataset loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')

Dataset loaded: (1470, 36)
Columns: ['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'AgeGroup']


In [3]:
df_model = df.copy()

gender_series = df_model['Gender'].copy()
age_group_series = df_model['AgeGroup'].astype(str).copy()
marital_series = df_model['MaritalStatus'].copy()

drop_cols = ['EmployeeNumber', 'EmployeeCount', 'StandardHours', 'Over18', 'AgeGroup']
df_model = df_model.drop(columns=[c for c in drop_cols if c in df_model.columns])


df_model['Attrition'] = (df_model['Attrition'] == 'Yes').astype(int)


le = LabelEncoder()
cat_cols = df_model.select_dtypes(include='object').columns
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

X = df_model.drop('Attrition', axis=1)
y = df_model['Attrition']

print(f'Features: {X.shape[1]}')
print(f'Target distribution: {y.value_counts().to_dict()}')
print(f'Attrition rate: {y.mean()*100:.1f}%')

Features: 30
Target distribution: {0: 1233, 1: 237}
Attrition rate: 16.1%


In [4]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


gender_test = gender_series.iloc[X_test.index]
age_test = age_group_series.iloc[X_test.index]
marital_test = marital_series.iloc[X_test.index]

print(f'Train size: {len(X_train)}')
print(f'Test size:  {len(X_test)}')

Train size: 1176
Test size:  294


In [5]:

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('=== LOGISTIC REGRESSION ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.1f}%')
print(classification_report(y_test, y_pred_lr, target_names=['Stay','Leave']))

joblib.dump(lr, '../models/logistic_regression.pkl')
print('Model saved: models/logistic_regression.pkl')

=== LOGISTIC REGRESSION ===
Accuracy: 86.7%
              precision    recall  f1-score   support

        Stay       0.87      0.98      0.93       247
       Leave       0.75      0.26      0.38        47

    accuracy                           0.87       294
   macro avg       0.81      0.62      0.65       294
weighted avg       0.85      0.87      0.84       294

Model saved: models/logistic_regression.pkl


In [7]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== RANDOM FOREST ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.1f}%')
print(classification_report(y_test, y_pred_rf, target_names=['Stay','Leave']))

joblib.dump(rf, '../models/random_forest.pkl')
print('Model saved: models/random_forest.pkl')

y_pred = y_pred_rf
print('\nUsing Random Forest predictions for fairness analysis')

=== RANDOM FOREST ===
Accuracy: 84.4%
              precision    recall  f1-score   support

        Stay       0.86      0.98      0.91       247
       Leave       0.55      0.13      0.21        47

    accuracy                           0.84       294
   macro avg       0.70      0.55      0.56       294
weighted avg       0.81      0.84      0.80       294

Model saved: models/random_forest.pkl

Using Random Forest predictions for fairness analysis


In [8]:
print('=== METRIC 1: DEMOGRAPHIC PARITY DIFFERENCE ===')
print('Measures: gap in prediction rates between groups')
print('Pass threshold: value close to 0 (< 0.1 acceptable)')
print()

dpd_gender = demographic_parity_difference(y_test, y_pred, sensitive_features=gender_test)
dpd_age = demographic_parity_difference(y_test, y_pred, sensitive_features=age_test)
dpd_marital = demographic_parity_difference(y_test, y_pred, sensitive_features=marital_test)

def flag(val, threshold=0.1):
    if abs(val) < threshold: return 'PASS'
    elif abs(val) < 0.2: return 'BORDERLINE'
    else: return 'FAIL'

print(f'Gender:         {dpd_gender:.4f}  [{flag(dpd_gender)}]')
print(f'Age Group:      {dpd_age:.4f}  [{flag(dpd_age)}]')
print(f'Marital Status: {dpd_marital:.4f}  [{flag(dpd_marital)}]')

=== METRIC 1: DEMOGRAPHIC PARITY DIFFERENCE ===
Measures: gap in prediction rates between groups
Pass threshold: value close to 0 (< 0.1 acceptable)

Gender:         0.0379  [PASS]
Age Group:      1.0000  [FAIL]
Marital Status: 0.0722  [PASS]


In [9]:
print('=== METRIC 2: EQUALIZED ODDS DIFFERENCE ===')
print('Measures: gap in both TPR and FPR between groups')
print('Pass threshold: value close to 0 (< 0.1 acceptable)')
print()

eod_gender = equalized_odds_difference(y_test, y_pred, sensitive_features=gender_test)
eod_age = equalized_odds_difference(y_test, y_pred, sensitive_features=age_test)
eod_marital = equalized_odds_difference(y_test, y_pred, sensitive_features=marital_test)

print(f'Gender:         {eod_gender:.4f}  [{flag(eod_gender)}]')
print(f'Age Group:      {eod_age:.4f}  [{flag(eod_age)}]')
print(f'Marital Status: {eod_marital:.4f}  [{flag(eod_marital)}]')

=== METRIC 2: EQUALIZED ODDS DIFFERENCE ===
Measures: gap in both TPR and FPR between groups
Pass threshold: value close to 0 (< 0.1 acceptable)

Gender:         0.0500  [PASS]
Age Group:      1.0000  [FAIL]
Marital Status: 0.1600  [BORDERLINE]


In [12]:
print('=== METRIC 3: PER-GROUP METRIC BREAKDOWN (MetricFrame) ===')
print('Shows selection rate, recall, precision per demographic group')
print()

metrics_dict = {
    'selection_rate': selection_rate,
    'recall': recall_score,
    'precision': precision_score
}


mf_gender = MetricFrame(
    metrics=metrics_dict,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=gender_test
)
print('--- BY GENDER ---')
print(mf_gender.by_group.round(3))
print(f'\nDifference (max-min):')
print(mf_gender.difference().round(3))


mf_age = MetricFrame(
    metrics=metrics_dict,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=age_test
)
print('\n--- BY AGE GROUP ---')
print(mf_age.by_group.round(3))
print(f'\nDifference (max-min):')
print(mf_age.difference().round(3))

=== METRIC 3: PER-GROUP METRIC BREAKDOWN (MetricFrame) ===
Shows selection rate, recall, precision per demographic group

--- BY GENDER ---
        selection_rate  recall  precision
Gender                                   
Female           0.060   0.125      0.286
Male             0.022   0.129      1.000

Difference (max-min):
selection_rate    0.038
recall            0.004
precision         0.714
dtype: float64

--- BY AGE GROUP ---
          selection_rate  recall  precision
AgeGroup                                   
18-30               0.12   0.333      0.667
31-40               0.00   0.000      0.000
41-50               0.00   0.000      0.000
51-60               0.00   0.000      0.000
nan                 1.00   0.000      0.000

Difference (max-min):
selection_rate    1.000
recall            0.333
precision         0.667
dtype: float64


In [13]:
print('=== METRIC 4: DISPARATE IMPACT RATIO ===')
print('Legal standard: ratio must be >= 0.8 (the 4/5ths rule)')
print('Below 0.8 = potential illegal discrimination')
print()

def disparate_impact(y_true, y_pred, sensitive_features):
    mf = MetricFrame(
        metrics=selection_rate,
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive_features
    )
    rates = mf.by_group
    min_rate = rates.min()
    max_rate = rates.max()
    ratio = min_rate / max_rate if max_rate > 0 else 1.0
    return ratio, rates

def di_flag(ratio):
    if ratio >= 0.8: return 'PASS'
    elif ratio >= 0.6: return 'BORDERLINE'
    else: return 'FAIL'

di_gender, rates_gender = disparate_impact(y_test, y_pred, gender_test)
di_age, rates_age = disparate_impact(y_test, y_pred, age_test)
di_marital, rates_marital = disparate_impact(y_test, y_pred, marital_test)

print(f'Gender Disparate Impact:         {di_gender:.4f}  [{di_flag(di_gender)}]')
print(f'  Selection rates: {rates_gender.round(3).to_dict()}')
print(f'\nAge Group Disparate Impact:      {di_age:.4f}  [{di_flag(di_age)}]')
print(f'  Selection rates: {rates_age.round(3).to_dict()}')
print(f'\nMarital Status Disparate Impact: {di_marital:.4f}  [{di_flag(di_marital)}]')
print(f'  Selection rates: {rates_marital.round(3).to_dict()}')

=== METRIC 4: DISPARATE IMPACT RATIO ===
Legal standard: ratio must be >= 0.8 (the 4/5ths rule)
Below 0.8 = potential illegal discrimination

Gender Disparate Impact:         0.3724  [FAIL]
  Selection rates: {'Female': 0.06, 'Male': 0.022}

Age Group Disparate Impact:      0.0000  [FAIL]
  Selection rates: {'18-30': 0.12, '31-40': 0.0, '41-50': 0.0, '51-60': 0.0, 'nan': 1.0}

Marital Status Disparate Impact: 0.0000  [FAIL]
  Selection rates: {'Divorced': 0.0, 'Married': 0.03, 'Single': 0.072}


In [14]:
print('=== METRIC 5: EQUAL OPPORTUNITY (True Positive Rate Parity) ===')
print('Measures: does model correctly identify attrition equally across groups?')
print('A model that misses attrition signals for one group fails this.')
print()

tpr_metrics = {'recall': recall_score}

mf_tpr_gender = MetricFrame(
    metrics=tpr_metrics,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=gender_test
)
mf_tpr_age = MetricFrame(
    metrics=tpr_metrics,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=age_test
)

tpr_diff_gender = mf_tpr_gender.difference()['recall']
tpr_diff_age = mf_tpr_age.difference()['recall']

print('TPR by Gender:')
print(mf_tpr_gender.by_group.round(3))
print(f'TPR difference: {tpr_diff_gender:.4f}  [{flag(tpr_diff_gender)}]')

print('\nTPR by Age Group:')
print(mf_tpr_age.by_group.round(3))
print(f'TPR difference: {tpr_diff_age:.4f}  [{flag(tpr_diff_age)}]')

=== METRIC 5: EQUAL OPPORTUNITY (True Positive Rate Parity) ===
Measures: does model correctly identify attrition equally across groups?
A model that misses attrition signals for one group fails this.

TPR by Gender:
        recall
Gender        
Female   0.125
Male     0.129
TPR difference: 0.0040  [PASS]

TPR by Age Group:
          recall
AgeGroup        
18-30      0.333
31-40      0.000
41-50      0.000
51-60      0.000
nan        0.000
TPR difference: 0.3333  [FAIL]


In [15]:
metrics_summary = {
    'Demographic\nParity (Gender)': (abs(dpd_gender), 0.1),
    'Demographic\nParity (Age)': (abs(dpd_age), 0.1),
    'Equalized\nOdds (Gender)': (abs(eod_gender), 0.1),
    'Equalized\nOdds (Age)': (abs(eod_age), 0.1),
    'Disparate\nImpact (Age)': (1 - di_age, 0.2),
}

labels = list(metrics_summary.keys())
values = [v[0] for v in metrics_summary.values()]
thresholds = [v[1] for v in metrics_summary.values()]
colors = ['#E24B4A' if v > t else '#1D9E75' for v, t in zip(values, thresholds)]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.5)
for i, (bar, t) in enumerate(zip(bars, thresholds)):
    ax.axhline(y=t, xmin=(i/len(labels))+0.05, xmax=((i+1)/len(labels))-0.05,
               color='orange', linewidth=2, linestyle='--')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

ax.set_title('Fairness Metrics Scorecard — Random Forest Model', fontsize=14, fontweight='bold')
ax.set_ylabel('Metric Value (lower = fairer)')
ax.set_ylim(0, max(values) * 1.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1D9E75', label='PASS'),
    Patch(facecolor='#E24B4A', label='FAIL'),
    Patch(facecolor='orange', label='Threshold')
]
ax.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.savefig('../reports/charts/04_fairness_scorecard.png', dpi=150, bbox_inches='tight')
print('Chart saved: 04_fairness_scorecard.png')
plt.show()

Chart saved: 04_fairness_scorecard.png


In [16]:
report = {
    'model': 'Random Forest',
    'dataset': 'IBM HR Attrition',
    'total_records': int(len(df)),
    'test_records': int(len(X_test)),
    'model_accuracy': round(accuracy_score(y_test, y_pred) * 100, 1),
    'metrics': {
        'demographic_parity_gender': {
            'value': round(float(dpd_gender), 4),
            'status': flag(dpd_gender),
            'threshold': 0.1,
            'description': 'Gap in attrition prediction rate between Male and Female employees'
        },
        'demographic_parity_age': {
            'value': round(float(dpd_age), 4),
            'status': flag(dpd_age),
            'threshold': 0.1,
            'description': 'Gap in attrition prediction rate across age groups'
        },
        'equalized_odds_gender': {
            'value': round(float(eod_gender), 4),
            'status': flag(eod_gender),
            'threshold': 0.1,
            'description': 'Combined TPR and FPR gap between Male and Female'
        },
        'equalized_odds_age': {
            'value': round(float(eod_age), 4),
            'status': flag(eod_age),
            'threshold': 0.1,
            'description': 'Combined TPR and FPR gap across age groups'
        },
        'disparate_impact_gender': {
            'value': round(float(di_gender), 4),
            'status': di_flag(di_gender),
            'threshold': 0.8,
            'description': 'Legal 4/5ths rule: ratio of selection rates between gender groups'
        },
        'disparate_impact_age': {
            'value': round(float(di_age), 4),
            'status': di_flag(di_age),
            'threshold': 0.8,
            'description': 'Legal 4/5ths rule: ratio of selection rates across age groups'
        },
        'disparate_impact_marital': {
            'value': round(float(di_marital), 4),
            'status': di_flag(di_marital),
            'threshold': 0.8,
            'description': 'Legal 4/5ths rule: ratio of selection rates across marital status'
        },
        'equal_opportunity_gender': {
            'value': round(float(tpr_diff_gender), 4),
            'status': flag(tpr_diff_gender),
            'threshold': 0.1,
            'description': 'Gap in True Positive Rate (recall) between Male and Female'
        },
        'equal_opportunity_age': {
            'value': round(float(tpr_diff_age), 4),
            'status': flag(tpr_diff_age),
            'threshold': 0.1,
            'description': 'Gap in True Positive Rate (recall) across age groups'
        }
    },
    'data_audit': {
        'gender_gap_pct': 2.2,
        'age_gap_pct': 14.8,
        'marital_gap_pct': 15.4,
        'biggest_bias': 'Marital Status (Single vs Divorced: 15.4% gap)'
    },
    'fix_recommendations': {
        'FAIL': [
            {
                'issue': 'Age group bias in predictions',
                'fix': 'Apply Reweighing — assign higher sample weights to underrepresented age groups during training',
                'library': 'aif360.algorithms.preprocessing.Reweighing'
            },
            {
                'issue': 'Marital status bias in selection rate',
                'fix': 'Remove MaritalStatus as a feature OR apply ThresholdOptimizer post-processing',
                'library': 'fairlearn.postprocessing.ThresholdOptimizer'
            }
        ],
        'BORDERLINE': [
            {
                'issue': 'Gender parity borderline',
                'fix': 'Monitor over time. Apply SMOTE oversampling if Female representation drops below 30%',
                'library': 'imblearn.over_sampling.SMOTE'
            }
        ]
    }
}

with open('../reports/bias_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print('bias_report.json saved to reports/')
print(f'\n=== SUMMARY ===')
fail_count = sum(1 for m in report['metrics'].values() if m['status'] == 'FAIL')
border_count = sum(1 for m in report['metrics'].values() if m['status'] == 'BORDERLINE')
pass_count = sum(1 for m in report['metrics'].values() if m['status'] == 'PASS')
print(f'PASS:       {pass_count} metrics')
print(f'BORDERLINE: {border_count} metrics')
print(f'FAIL:       {fail_count} metrics')
print(f'\nModel accuracy: {report["model_accuracy"]}%')
print('\n=== MODEL TRAINING + FAIRNESS METRICS COMPLETE ===')
print('Next: build the Flask backend (backend/app.py)')

bias_report.json saved to reports/

=== SUMMARY ===
PASS:       3 metrics
BORDERLINE: 0 metrics
FAIL:       6 metrics

Model accuracy: 84.4%

=== MODEL TRAINING + FAIRNESS METRICS COMPLETE ===
Next: build the Flask backend (backend/app.py)
